# Verdict in 3 minutes

Small, fast, honest decision models. This notebook runs on the free Colab CPU.

Repo: https://github.com/Manavarya09/verdict

In [ ]:
!pip -q install verdictml

## 1. Zero-shot: choose, score, check

In [ ]:
from verdict import Verdict
v = Verdict()   # downloads verdict-small (118M, multilingual) the first time

ticket = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
print(v.choose(ticket, {"billing": "invoices, payments, refunds", "technical": "bugs, outages", "sales": "pricing, new contracts", "other": "everything else"}, prompt="What does the user want?"))
print(v.score(ticket, scale=(1, 3), rubric={1: "not urgent", 2: "soon", 3: "blocking issue or hard deadline"}, prompt="How urgent is this?"))
print(v.check(ticket, claim="the customer asks for a refund"))

## 2. Teach it your decision in seconds, then calibrate

A handful of labelled examples per class is enough for the prototype head. `calibrate()` on held-out rows gives honest probabilities and an abstain flag with a coverage guarantee.

In [ ]:
from verdict import Question
from verdict.types import options_from

q = Question(kind="choose", options=options_from(["refund", "shipping", "account", "other"]))
d = v.compile(q)

train = [
    ("I want my money back", "refund"), ("charged twice, please reverse", "refund"), ("refund the duplicate charge", "refund"),
    ("where is my parcel", "shipping"), ("tracking says delivered but nothing arrived", "shipping"), ("my order is late", "shipping"),
    ("reset my password", "account"), ("change the email on my profile", "account"), ("I cannot log in", "account"),
    ("do you have a mobile app", "other"), ("what are your opening hours", "other"), ("is there a student discount", "other"),
] * 3
held_out = [("send my money back now", "refund"), ("the courier lost it", "shipping"), ("locked out of my account", "account"), ("hello?", "other")] * 6

d.fit(train)
d.calibrate(held_out, coverage=0.9)
print(d.evaluate(held_out))
for text in ["please refund me", "package never came", "who are you"]:
    a = d(text)
    print(f"{text!r:25} -> {a.label:9} p={a.confidence.probability:.2f} abstain={a.confidence.abstain} set={a.confidence.conformal_set}")

## 3. The cascade: Verdict answers what it is sure about, your LLM gets the rest

In [ ]:
from verdict.cascade import Cascade

route = Cascade(d, fallback=lambda text, answer: f"LLM would decide among {answer.confidence.conformal_set}")
for text in ["please refund me", "hmm, something about my thing"]:
    print(text, "->", route(text))
print(route.stats())

## 4. Reproduce a benchmark row

The README numbers come from `bench/`. This runs Banking77 zero-shot and 16-shot on 500 test rows (a few minutes on CPU).

In [ ]:
!pip -q install "verdictml[bench]"
!git clone -q https://github.com/Manavarya09/verdict /content/verdict
%cd /content/verdict
!python -m bench.run banking77 --n 500 --shots 0,16